# Comparing Feature Selection Techniques

Practice activity from Microsoft *Foundations of AI and Machine Learning* — Module: AI/ML Algorithms and Techniques.

Apply three feature-selection techniques to the same dataset and compare what each one keeps:
1. **Backward elimination** — drop the feature with the highest OLS p-value, repeat until all surviving p-values < 0.05.
2. **Forward selection** — greedily add the feature that most improves R-squared.
3. **LASSO** — L1 regularization shrinks redundant feature coefficients to zero.

## 2. Imports

In [1]:
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

## 3. Load the dataset

In [2]:
data = {
    'StudyHours':    [1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
    'PrevExamScore': [30, 40, 45, 50, 60, 65, 70, 75, 80, 85],
    'Pass':          [0,  0,  0,  0,  0,  1,  1,  1,  1,  1],
}
df = pd.DataFrame(data)

X = df[['StudyHours', 'PrevExamScore']]
y = df['Pass']
df

,StudyHours,PrevExamScore,Pass
0,1,30,0
1,2,40,0
2,3,45,0
3,4,50,0
4,5,60,0
5,6,65,1
6,7,70,1
7,8,75,1
8,9,80,1
9,10,85,1


### Spot the collinearity

Before running any selection method, look at the correlation between features. If two features are nearly perfectly correlated, *which* one each method keeps becomes essentially arbitrary.

In [3]:
X.corr()

,StudyHours,PrevExamScore
StudyHours,1.000000,0.994987
PrevExamScore,0.994987,1.000000


## 4. Backward elimination

Fit OLS with all features, look at p-values, drop the worst one if it's above 0.05, repeat.

In [4]:
initial_fit = sm.OLS(y, sm.add_constant(X)).fit()
print(initial_fit.summary().tables[1])

                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.3333      1.464     -0.228      0.826      -3.796       3.129
StudyHours        0.1515      0.324      0.468      0.654      -0.615       0.918
PrevExamScore -3.053e-16      0.054  -5.68e-15      1.000      -0.127       0.127


In [5]:
def backward_elimination(X, y, threshold=0.05):
    features = list(X.columns)
    while features:
        X_const = sm.add_constant(X[features], has_constant='add')
        model = sm.OLS(y, X_const).fit()
        pvals = model.pvalues.drop('const', errors='ignore')
        worst_pval = pvals.max()
        if worst_pval > threshold:
            worst_feature = pvals.idxmax()
            print(f'  Drop {worst_feature!r:18s}  p-value = {worst_pval:.4f}')
            features.remove(worst_feature)
        else:
            break
    return features

be_features = backward_elimination(X, y)
print(f'\nSelected features (backward elimination): {be_features}')

  Drop 'PrevExamScore'     p-value = 1.0000

Selected features (backward elimination): ['StudyHours']


## 5. Forward selection

Greedy: each round, try every remaining feature, score it on a held-out split, keep the one with the best R-squared. Stop when no candidate improves the score.

In [6]:
def forward_selection(X, y):
    remaining = set(X.columns)
    selected = []
    best_score = 0.0

    while remaining:
        round_scores = []
        for feature in remaining:
            trial = selected + [feature]
            X_train, X_test, y_train, y_test = train_test_split(
                X[trial], y, test_size=0.2, random_state=42
            )
            m = LinearRegression().fit(X_train, y_train)
            round_scores.append((r2_score(y_test, m.predict(X_test)), feature))

        round_scores.sort(reverse=True)
        top_score, top_feature = round_scores[0]

        if top_score > best_score:
            selected.append(top_feature)
            remaining.remove(top_feature)
            best_score = top_score
            print(f'  Added {top_feature!r:18s}  R-squared = {top_score:.4f}')
        else:
            break

    return selected, best_score

fs_features, fs_score = forward_selection(X, y)
print(f'\nSelected features (forward selection): {fs_features}')

  Added 'PrevExamScore'     R-squared = 1.0000

Selected features (forward selection): ['PrevExamScore']


## 6. LASSO

L1 regularization: any feature whose coefficient gets driven to exactly zero is effectively dropped.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
lasso = Lasso(alpha=0.1).fit(X_train, y_train)
lasso_r2 = r2_score(y_test, lasso.predict(X_test))

print(f'R-squared: {lasso_r2:.4f}')
for name, coef in zip(X.columns, lasso.coef_):
    status = 'kept' if coef != 0 else 'dropped (shrunk to 0)'
    print(f'  {name:14s} coef={coef:+.6f}  [{status}]')

lasso_features = [name for name, coef in zip(X.columns, lasso.coef_) if coef != 0]

R-squared: 0.9998
  StudyHours     coef=+0.000000  [dropped (shrunk to 0)]
  PrevExamScore  coef=+0.024636  [kept]


## Comparison

In [8]:
print('Selected features by method:')
print(f'  Backward elimination: {be_features}')
print(f'  Forward selection:    {fs_features}')
print(f'  LASSO (alpha=0.1):    {lasso_features}')

Selected features by method:
  Backward elimination: ['StudyHours']
  Forward selection:    ['PrevExamScore']
  LASSO (alpha=0.1):    ['PrevExamScore']


## Key takeaways

**The methods disagree on which feature to keep — and that disagreement is the point.** `StudyHours` and `PrevExamScore` are nearly perfectly correlated (look at the correlation matrix above), so picking "the right" one isn't really well-defined; any single feature suffices.

| Method | Kept here | What it's sensitive to |
|---|---|---|
| Backward elimination | `StudyHours` | OLS assumes features aren't perfectly collinear. When they are, the model puts all the weight on whichever feature is listed first, gives the other a p-value of 1.0, and drops it. Fragile under multicollinearity. |
| Forward selection | `PrevExamScore` | Depends on the train/test split (`random_state`) and the order in which features are tried. Stable here, but with a different split it could flip. |
| LASSO | `PrevExamScore` | L1 picks one of a set of correlated features essentially arbitrarily. **Elastic Net** is the standard fix when you want correlated features to survive as a group. |

**Practical workflow on real data:**
1. Check pairwise correlations and VIF *before* running selection.
2. If two features are essentially duplicates, drop one by domain knowledge.
3. Then run one (or more!) selection methods. Cross-validate the final feature set rather than trusting a single split.
4. Treat p-values, R² improvements, and LASSO coefficients as *signals*, not verdicts.